In [ ]:
!pip -q install google-api-python-client isodate requests

In [ ]:
import pandas as pd
import re, json, time, random, requests
from datetime import datetime, timezone
from collections import Counter
from googleapiclient.discovery import build
import isodate
from google.colab import files

API_KEY = "YOUTUBE_API_KEY"

# ---------- helpers ----------
def humanize_views(n):
    if n is None: return None
    n = float(n)
    if n < 1000: return f"{int(n)} views"
    for unit, div in [("K",1e3),("M",1e6),("B",1e9),("T",1e12)]:
        if n < div*1000:
            val = n/div
            return (f"{val:.1f}{unit} views" if val < 10 else f"{val:.0f}{unit} views")
    return f"{n:.0f} views"

def humanize_duration(sec):
    if sec is None: return None
    sec = int(sec)
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h}:{m:02d}:{s:02d}" if h>0 else f"{m}:{s:02d}"

# ---------- political coding (binary only) ----------
def clean_for_coding(text: str) -> str:
    if not text:
        return ""
    text = str(text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[@#]\w+", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

ENTERTAINMENT_HINTS = [
    "lyrics","lyric","letra","mv","music video","official video","official music video",
    "trailer","teaser","in theatres","in theaters","full movie",
    "gameplay","walkthrough","let's play","lets play","speedrun",
    "episode","part","pt","pokemon","grumplocke",
]

POLITICAL_ANCHORS = [
    "election","vote","voting","ballot","campaign","primary","debate","polls",
    "congress","senate","house of representatives","white house","capitol",
    "supreme court","scotus","federal court",
    "president","governor","mayor","attorney general",
    "doj","department of justice","fbi","dhs","state department","pentagon",
    "immigration","asylum","deportation","border",
    "sanction","nato","ukraine","russia","israel","gaza","taiwan",
]

POLITICAL_STRONG = [
    "election","vote","voting","ballot","campaign","primary","caucus",
    "debate","polls","rally","electoral","democrat","republican","gop",
    "congress","senate","house of representatives","white house","capitol",
    "supreme court","scotus","federal court","president","governor","mayor","attorney general",
    "doj","department of justice","fbi","dhs","state department","pentagon",
    "immigration","asylum","deportation","border",
    "executive order","regulation","sanction","tariff",
    "nato","ukraine","russia","israel","gaza","china","taiwan","iran",
    "budget","federal reserve","fed","interest rate",
]

POLITICAL_WEAK = ["law","policy","bill","ban","tax","inflation"]

def _compile_word_patterns(words):
    parts = []
    for w in words:
        w = w.strip().lower()
        if not w:
            continue
        if re.fullmatch(r"[a-z]{2,4}", w):
            parts.append(rf"(?<![a-z]){re.escape(w)}(?![a-z])")
        else:
            parts.append(rf"\b{re.escape(w)}\b")
    return re.compile("(" + "|".join(parts) + ")", flags=re.IGNORECASE) if parts else re.compile(r"$^")

ENT_PAT = _compile_word_patterns(ENTERTAINMENT_HINTS)
ANCHOR_PAT = _compile_word_patterns(POLITICAL_ANCHORS)
STRONG_PAT = _compile_word_patterns(POLITICAL_STRONG)
WEAK_PAT   = _compile_word_patterns(POLITICAL_WEAK)

def code_political(title, description):
    text = clean_for_coding((title or "") + " " + (description or ""))
    if not text:
        return 0

    if STRONG_PAT.search(text):
        return 1

    if ENT_PAT.search(text):
        return 0

    if ANCHOR_PAT.search(text) and WEAK_PAT.search(text):
        return 1

    return 0

# ---------- main ----------
def fetch_us_daily_mostpopular(api_key, top_n=100):
    yt = build("youtube", "v3", developerKey=api_key)
    snapshot = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    rows = []
    token = None
    fetched = 0

    while fetched < top_n:
        batch = min(50, top_n - fetched)
        resp = yt.videos().list(
            part="snippet,statistics,contentDetails",
            chart="mostPopular",
            regionCode="US",
            maxResults=batch,
            pageToken=token
        ).execute()

        items = resp.get("items", [])
        if not items:
            break

        for it in items:
            sn = it.get("snippet", {})
            st = it.get("statistics", {})
            ct = it.get("contentDetails", {})
            vid = it.get("id")

            pub = sn.get("publishedAt")
            pub_text = None
            if pub:
                dt = datetime.fromisoformat(pub.replace("Z", "+00:00"))
                pub_text = dt.strftime("%Y-%m-%d %H:%M UTC")

            dur_iso = ct.get("duration")
            dur_sec = isodate.parse_duration(dur_iso).total_seconds() if dur_iso else None

            thumbs = {k: v.get("url") for k, v in (sn.get("thumbnails") or {}).items()
                      if isinstance(v, dict) and v.get("url")}

            views = int(st["viewCount"]) if "viewCount" in st else None

            political = code_political(sn.get("title"), sn.get("description"))

            rows.append({
                "snapshotDate": snapshot,
                "title": sn.get("title"),
                "description": sn.get("description"),
                "publishedDate": pub,
                "publishedText": pub_text,
                "videoId": vid,
                "videoUrl": f"https://www.youtube.com/watch?v={vid}",
                "channelName": sn.get("channelTitle"),
                "channelId": sn.get("channelId"),
                "channelUrl": f"https://www.youtube.com/channel/{sn.get('channelId')}" if sn.get("channelId") else None,
                "thumbnails": json.dumps(thumbs, ensure_ascii=False),
                "views": views,
                "viewsText": humanize_views(views),
                "duration": dur_iso,
                "durationText": humanize_duration(dur_sec),
                "political": political
            })

        fetched += len(items)
        token = resp.get("nextPageToken")
        if not token:
            break

    df = pd.DataFrame(rows).head(top_n)

    # ✅ daily political share（同一天在每一行重复显示）
    df["daily_political_share"] = df.groupby("snapshotDate")["political"].transform("mean")

    final_cols = [
        "snapshotDate",
        "title","description","publishedDate","publishedText",
        "videoId","videoUrl",
        "channelName","channelId","channelUrl",
        "thumbnails",
        "views","viewsText",
        "duration","durationText",
        "political","daily_political_share"
    ]
    df = df[final_cols]

    return df, snapshot

# ---------- run ----------
df, snap = fetch_us_daily_mostpopular(API_KEY, top_n=100)

out = f"US_mostPopular_{snap}.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

print("Saved:", out)
print("Daily political share:", float(df["daily_political_share"].iloc[0]))

files.download(out)


Saved: US_mostPopular_2026-02-10.csv
Daily political share: 0.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>